# 03 — Model Training

Trains two credit risk models and saves them for deployment.

**Models:**
- Logistic Regression (interpretable baseline — FICO-style scorecard logic)
- LightGBM (gradient boosting — state-of-the-art for tabular credit data)

**Key decisions:**
- Temporal train/test split (block 17,000,000 ≈ April 2023) to prevent future leakage
- Class imbalance handled via `class_weight='balanced'` and `scale_pos_weight`
- Probability calibration via Platt scaling — critical for reliable PD estimates

**Why temporal split?**  
Random splits would allow wallets that defaulted *after* the split to appear in training
via their earlier transaction history, leaking future information into the model.
Temporal splitting mirrors how the model would be deployed in production.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

from src.models.train import load_and_split, train_logistic_regression, train_lightgbm, save_model

sns.set_theme(style='whitegrid', context='notebook')
BRAND_BLUE = '#185FA5'

## 1. Load and split dataset

In [ ]:
FEATURE_PATH = Path('../data/processed/feature_matrix.parquet')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

X_train, X_test, y_train, y_test, feature_cols = load_and_split(FEATURE_PATH)

print(f'Train: {len(X_train):,} wallets | Default rate: {y_train.mean():.2%}')
print(f'Test:  {len(X_test):,} wallets  | Default rate: {y_test.mean():.2%}')
print(f'Features: {len(feature_cols)}')

## 2. Train Logistic Regression (baseline)

Why LR as baseline?
- Produces log-odds scores that map directly to FICO-style scorecards
- Feature coefficients are directly interpretable
- Used in traditional bank credit models — familiar to hiring managers in banking

In [ ]:
lr_model = train_logistic_regression(X_train, y_train)
lr_prob = lr_model.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
print(f'Logistic Regression — Test ROC-AUC: {lr_auc:.4f}')

save_model(lr_model, MODELS_DIR / 'logistic_regression.pkl')

## 3. Train LightGBM

Why LightGBM?
- State-of-the-art for tabular credit data (wins most Kaggle credit competitions)
- Handles missing values natively
- Faster than XGBoost at similar accuracy
- SHAP-compatible for full explainability

In [ ]:
lgb_model = train_lightgbm(X_train, y_train)
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_prob)
print(f'LightGBM — Test ROC-AUC: {lgb_auc:.4f}')

save_model(lgb_model, MODELS_DIR / 'lightgbm.pkl')

## 4. Quick performance comparison

In [ ]:
from sklearn.metrics import brier_score_loss
from src.models.evaluate import ks_statistic, gini_coefficient, lift_at_decile

results = []
for name, prob in [('Logistic Regression', lr_prob), ('LightGBM', lgb_prob)]:
    results.append({
        'Model': name,
        'ROC-AUC': roc_auc_score(y_test.values, prob),
        'KS': ks_statistic(y_test.values, prob),
        'Gini': gini_coefficient(y_test.values, prob),
        'Brier': brier_score_loss(y_test.values, prob),
        'Lift@D1': lift_at_decile(y_test.values, prob, 1),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4).to_string())

## 5. Save metadata

In [ ]:
meta = {
    'feature_columns': feature_cols,
    'temporal_split_block': 17_000_000,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'train_default_rate': float(y_train.mean()),
    'test_default_rate': float(y_test.mean()),
    'lr_roc_auc': lr_auc,
    'lgb_roc_auc': lgb_auc,
}

with (MODELS_DIR / 'feature_columns.json').open('w') as f:
    json.dump(feature_cols, f, indent=2)
with (MODELS_DIR / 'training_metadata.json').open('w') as f:
    json.dump(meta, f, indent=2)

print('Metadata saved.')
print(f'Models directory: {MODELS_DIR.resolve()}')

## Next step

Proceed to `04_model_evaluation.ipynb` for the full evaluation suite with plots.